# Process ARCH runs (post Sequentify docker output)

## Imports and paths definitions
Prepare the 'All_Samples/5_vcfs' folder, and sample_ifn should list sample names to process under a column named 'Sample'.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os,sys
import pickle
import shutil
import subprocess
from pathlib import Path

# Panel: despite the name (arch/geno), support also geno (relevant to run geno till the QC step)
panel_type = 'arch' # geno / arch

# Run name (will be used to name output files)
seq_run = 'DS_2M_RG_VS_3X'
# seq_run = 'PERIBLOOD_merged_arch'

# Folder containing a sub-folder per sample, output files will be saved here
arch_dir = f"/home/projects/shlush/shared/runs_analysis/{seq_run}/analysis_{panel_type}"

output_base_dir = f"{arch_dir}/All_Samples/"
output_5_vcfs_base_dir = f"{output_base_dir}/5_vcfs"

# Directory containing *_R{1,2}.fastq.gz files for this run
fastq_dir = f"/home/projects/shlush/shared/runs_analysis/{seq_run}/fastq_PERIBLOOD"

# Root directory of per-sample BAM sub-folders for this run
mapped_dir = f"{output_base_dir}/2_mapped"

# Merged varscan of all samples
merged_varscan_file_path = os.path.join(output_5_vcfs_base_dir, "merged_varscan_annotations.txt")

# File listing sample names to process
samples_for_processing_ifn = f"{panel_type}_{seq_run}_samples_for_processing.csv"
all_samples_classified_ifn = f"{panel_type}_{seq_run}_all_classified.csv"

# Rebuild flag, if False, will not rerun steps that already have output files
rebuild = False

# Define the path to Rscript and R script (update with your own)
rscript_exe = "/home/projects/shlush/USER/anaconda3/envs/r_env/bin/Rscript"
conda_env = "r_env"

# output file names (sequential filtering order)
listed_output_path = os.path.join(output_5_vcfs_base_dir, f"FilterMutations_{seq_run}_arch_py_listed.tsv")
not_listed_output_path = os.path.join(output_5_vcfs_base_dir, f"FilterMutations_{seq_run}_arch_py_filtered_yet_not_listed.tsv")
# Remove recurrent mutations (>=2 samples) that lack haematopoietic COSMIC / ClinVar
# support. Off by default: the COSMIC annotation is sparse (only ~1% of indels carry any),
# so the step also removes curated hotspots. When False the step is skipped and
# "_FilteredRecurrent" drops out of the downstream file names, so the filtered and
# unfiltered variants of a run can coexist on disk.
filter_recurrent_without_support = False
recurrence_tag = "_FilteredRecurrent" if filter_recurrent_without_support else ""

no_recurrent_output_path = os.path.join(output_5_vcfs_base_dir, f"FilterMutations_{seq_run}_arch_listed{recurrence_tag}.tsv")

# Filter by VAF
min_final_vaf = 0.05
gene_min_vaf_overrides = {'BRAF': 0.01}  # per-gene VAF threshold overrides; genes not listed use min_final_vaf
master_output_path = os.path.join(output_5_vcfs_base_dir, f"FilterMutations_{seq_run}_arch_listed{recurrence_tag}_VAF_{min_final_vaf}.tsv")

# CALR type-1 and type-2 results files
calr_type1_results_file = os.path.join(output_5_vcfs_base_dir, f"CALR_type1_results_{seq_run}.tsv")
calr_type2_results_file = os.path.join(output_5_vcfs_base_dir, f"CALR_type2_results_{seq_run}.tsv")

# Denver data
denver_base_dir = "/home/projects/shlush/shared/runs_analysis/NovaseqR99/embedded_demultiplexing/Analysis"

# Files post Denver filtering
master_output_with_denver_path = master_output_path.replace('.tsv', '_with_denver.tsv') 
denver_filtered_output_path = master_output_path.replace('.tsv', '_Denver_filtered.tsv')

def get_git_root():
    """Returns the root directory of the current git repository."""
    try:
        # Run the git command and capture the output
        root_path = subprocess.check_output(
            ['git', 'rev-parse', '--show-toplevel'], 
            text=True, 
            stderr=subprocess.DEVNULL
        ).strip()
        return Path(root_path)
    except subprocess.CalledProcessError:
        print("Not inside a Git repository or Git is not installed.")
        return None


## Create All_Samples/5_vcfs and 2_mapped folders (links to actual samples folders)

In [ ]:
import yaml
import re

sample_mapping = []

for sub_dir in ['2_mapped', '5_vcfs']:
    sub_dir_path = os.path.join(output_base_dir, sub_dir)
    os.makedirs(sub_dir_path, exist_ok=True)

# Iterate over all folders named job## under arch_dir
job_dirs = [d for d in os.listdir(arch_dir) if d.startswith("job")]
for job_num in job_dirs:
    if re.match(r'^job\d+$', job_num):
        sample_yaml_path = os.path.join(arch_dir, job_num, 'Sample_file_path_info.yaml')
        if os.path.exists(sample_yaml_path):
            with open(sample_yaml_path, 'r') as f:
                d = yaml.safe_load(f)
                sample_names = d.keys()
                for sample_name in sample_names:
                    for sub_dir in ['2_mapped', '5_vcfs']:
                        src_path = os.path.join(arch_dir, job_num, 'paired', sub_dir, sample_name)
                        dst_path = os.path.join(output_base_dir, sub_dir, sample_name)
                        
                        if not os.path.islink(dst_path):
                            print(f"Linking {sample_name} from {os.path.join(arch_dir, job_num, 'paired', sub_dir, sample_name)} to {os.path.join(output_base_dir, sub_dir, sample_name)}")
                            os.symlink(src_path, dst_path)

## Create QCs from statistics

Note that if sample names under Sample column are not in this format: EXPERIMENT_PANEL_SAMPLE_INDEX, e.g. NEX84_ARCH_PBMDS319_S297 (PANEL can be ARCH or Geno) the code below will break. 
Run it once to generate the merged statistics file, modify it such that Sample column follows the above format and rerun with rebuild = False so the modified file will be used and not overwritten.

Also note you'll need to look at the plots in order to update ***min_f_frag_x100*** and rerun if needed.

### Generate QC report

In [ ]:
import sys
import subprocess

prev_rebuild = rebuild
rebuild = False

# Prepare arguments for mip_run_statistics.py
mip_run_statistics_script = os.path.abspath(os.path.join('..', 'utils', 'mip_run_statistics.py'))

output_dir = output_5_vcfs_base_dir  # already defined above
seq_run_name = seq_run       # already defined above
runs_base_dir = "/home/projects/shlush/shared/runs_analysis"  # update if needed
# sample_fixed_column = "FixedSample"
sample_fixed_column = "Sample"

if panel_type == 'arch':
    min_lib_size = 70000 # Minimum estimated library size (used to decide if DNA re-extraction is needed)
    min_base_cov = 20 # Minimum per-base dedup coverage depth to consider a base as valid
    min_f_dedup_cov = 0.8 # Minimum fraction of valid bases to consider a sample as valid
    min_f_dedup_cov_to_process = 0.1 # process poor samples too to get their info
elif panel_type == 'geno':
    min_lib_size = 40000 # Minimum estimated library size (used to decide if DNA re-extraction is needed)
    min_base_cov = 4 # Minimum per-base dedup coverage depth to consider a base as valid
    min_f_dedup_cov = 0.3 # Minimum fraction of valid bases to consider a sample as valid
    min_f_dedup_cov_to_process = 0.1 # process poor samples too to get their info
    
# Build the command list
cmd = [
    sys.executable,
    mip_run_statistics_script,
    "--output-dir", output_dir,
    "--seq-run-name", seq_run_name,
    "--panel", panel_type,
    "--runs-base-dir", runs_base_dir,
    "--sample-fixed-column", sample_fixed_column,
    "--min-lib-size", str(min_lib_size),
    "--min-base-cov", str(min_base_cov),
    "--min-f-dedup-cov", str(min_f_dedup_cov),
]

if rebuild:
    cmd.append("--rebuild")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)

all_samples_df = pd.read_csv(os.path.join(output_5_vcfs_base_dir, all_samples_classified_ifn))
samples_for_processing_df = all_samples_df[all_samples_df[f'base >={min_base_cov} dedup %on-target cov'] >= min_f_dedup_cov_to_process]
samples_for_processing_df.to_csv(os.path.join(output_5_vcfs_base_dir, samples_for_processing_ifn))

rebuild = prev_rebuild

## Interactive vesion: Only run if a few samples, otherwise run the LSF version below

In [ ]:
valid_samples = pd.read_csv(os.path.join(output_5_vcfs_base_dir, samples_for_processing_ifn))

for sample_name in valid_samples['Sample']:  
    vcf_file = f"{output_5_vcfs_base_dir}/{sample_name}/{sample_name}.varscan_C8_AF0.001_Q15_norm.dedup.vcf.gz" 
    filtered_vcf_file = f"{output_5_vcfs_base_dir}/{sample_name}/{sample_name}.varscan_C8_AF0.001_Q15_norm.dedup.filtered.vcf.gz"
    
    output_anvr_varscan = f"{output_5_vcfs_base_dir}/{sample_name}/varscan_annot_dedup_filt" 
    
    # Invoke command if output file does not exist or if rebuild is True
    if not os.path.exists(f"{output_anvr_varscan}.hg38_multianno.txt") or rebuild:
        print(f"{sample_name}: Filtering VAF for actual mutations...")
    
        cmd = f"module load BCFtools; bcftools view -i 'ALT != \".\"'  {vcf_file} -Oz -o {filtered_vcf_file}"
        get_ipython().system('{cmd}')    
        
        # prev: perl = "/apps/RH7U2/gnu/perl/5.24.0/bin/perl"
        # prev: annovar = "/apps/RH7U2/general/annovar/2017Jun01/table_annovar.pl"
        # prev: human_db = "/home/projects/shlush/yardenab/humandb/"
        # prev: db_list = "refGene,gnomad211_genome,kaviar_20150923,cosmic70,avsnp150,clinvar_20170905"
        perl = "/usr/bin/perl"
        annovar = "/home/projects/shlush/shared/annovar/2022-08-02/table_annovar.pl"
        human_db = "/home/projects/shlush/shared/annovar/humandb/"
        db_list = "refGene,gnomad41_genome,kaviar_20150923,cosmic103_mapped,avsnp151,clinvar_20250721"

        hg = "hg38"
        operation = "g,f,f,f,f,f"
        
        cmd = "{perl} {annovar} {vcf_file} {human_db} -buildver {hg} -out {output} -remove -protocol {db_list} -operation {operation} -nastring . -vcfinput".format(annovar=annovar, 
        vcf_file=filtered_vcf_file, 
        output=output_anvr_varscan, 
        human_db=human_db, hg=hg, operation=operation, db_list=db_list, perl=perl)

        print(f"{sample_name}: Running ANNOVAR...")
        print(f"Running cmd: {cmd}")
        get_ipython().system('{cmd}')
    else:
        print(f"{sample_name}: Skipping ANNOVAR, output already exists.")



## LSF version: submit one job per sample (instead of iterative)

Run the cell below to submit LSF jobs for each valid sample. Each job runs `run_annovar_one_lsf.sh` with `BASE_DIR`, `SAMPLE_NAME`, and `REBUILD` set. Logs go to `{base_dir}/lsf_logs/`. Adjust `-q`, `-M`, `-W` if your cluster requires different queue/resources.

In [ ]:
# LSF: submit one job per sample (same logic as iterative cell above)
import subprocess

valid_samples = pd.read_csv(os.path.join(output_5_vcfs_base_dir, samples_for_processing_ifn))
# Script is next to this notebook (Gal_code/run_annovar_one_lsf.sh)
for candidate in [
    os.path.join(os.getcwd(), "run_annovar_one_lsf.sh"),
    os.path.join(os.path.dirname(output_5_vcfs_base_dir), "Gal_code", "run_annovar_one_lsf.sh"),
    "run_annovar_one_lsf.sh",
]:
    if os.path.exists(candidate):
        script_path = os.path.abspath(candidate)
        break
else:
    raise FileNotFoundError("run_annovar_one_lsf.sh not found")

log_dir = os.path.join(output_5_vcfs_base_dir, "lsf_logs")
os.makedirs(log_dir, exist_ok=True)
rebuild_val = "1" if rebuild else "0"

for sample_name in valid_samples['Sample']:
    job_name = f"annovar_{sample_name}"
    out_log = os.path.join(log_dir, f"{job_name}.out")
    err_log = os.path.join(log_dir, f"{job_name}.err")
    cmd = [
        "bsub",
        "-q", "short",
        "-J", job_name,
        "-o", out_log,
        "-e", err_log,
        "bash", script_path, output_5_vcfs_base_dir, sample_name, rebuild_val,
    ]
    subprocess.run(cmd, check=False)
    print(f"Submitted: {job_name}")

## Get list of varscan_annot_dedup.hg38_multianno.txt file sizes

Run when previous cell finishes

In [ ]:
# Target filename to search
target_filename = "varscan_annot_dedup_filt.hg38_multianno.txt"

records = []

# Iterate through all subfolders in base_dir
for entry in os.listdir(output_5_vcfs_base_dir):
    subfolder_path = os.path.join(output_5_vcfs_base_dir, entry)
    if os.path.isdir(subfolder_path):
        sample_name = entry  # sample name is the subfolder name
        file_path = os.path.join(subfolder_path, target_filename)
        if os.path.exists(file_path):
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)  # size in MB
            records.append({
                "Sample_Name": sample_name,
                "File_Name": target_filename,
                "File_Path": file_path,
                "File_Size_MB": round(file_size_mb, 2)
            })
        else:
            records.append({
                "Sample_Name": sample_name,
                "File_Name": None,
                "File_Path": None,
                "File_Size_MB": None
            })

# Create DataFrame
df = pd.DataFrame(records)

# Export to Excel
output_excel = os.path.join(output_5_vcfs_base_dir, "multianno_file_report.xlsx")
df.to_excel(output_excel, index=False)

print(f"✅ Report saved to: {output_excel}")
print(df.head())

## convert vcf to formatted format

In [ ]:
###this creates the function


import pandas as pd
from pandas.errors import EmptyDataError
import os

def varscan_multianno_to_df_no_filter_fast_old(filename):
    out_filename = filename.replace('.txt', '.formatted.txt')

    try:
        # read as all strings for speed
        df = pd.read_csv(filename, sep='\t', skiprows=1, header=None, dtype=str)
    except EmptyDataError:
        print(f"⚠️ Empty file: {filename}")
        return

    # get headers
    with open(filename, 'r') as f:
        txt_header = f.readline().strip().split('\t')

    vcf_file = filename.replace('.txt', '.vcf')
    vcf_header = []
    with open(vcf_file, 'r') as f:
        for line in f:
            if line.startswith('#CHROM'):
                vcf_header = line.strip().split('\t')
                break

    # assign columns
    print(f'txt_header ({len(txt_header)}): {txt_header}')
    print(f'vcf_header ({len(vcf_header)}): {vcf_header}')
    df.columns = txt_header + ['.', '_'] + vcf_header

    # find end of annotations
    end_of_annotation_idx = list(df.columns).index('Otherinfo1') - 1

    # vectorized parsing
    format_col = df['FORMAT']
    sample_col = df.iloc[:, -1]  # last column is sample
    format_keys = format_col.iloc[0].split(':')

    # build expanded sample
    sample_expanded = sample_col.str.split(':', expand=True)

    depth_col = sample_expanded.iloc[:, format_keys.index('DP')] if 'DP' in format_keys else '.'
    vaf_col = sample_expanded.iloc[:, format_keys.index('FREQ')] if 'FREQ' in format_keys else '.'

    # build final dataframe
    final_df = pd.concat([df.iloc[:, :end_of_annotation_idx], df['FILTER'], depth_col, vaf_col], axis=1)
    final_df.columns = list(final_df.columns[:-2]) + ['Depth', 'VAF']

    # save
    final_df.to_csv(out_filename, sep='\t', index=False)
    print(f"✅ Processed {os.path.basename(filename)} → {out_filename}")


def varscan_multianno_to_df_no_filter_fast(filename):
    out_filename = filename.replace('.txt', '.formatted.txt')

    try:
        # Read as all strings for speed. 
        # header=None prevents pandas from misaligning columns on load
        df = pd.read_csv(filename, sep='\t', skiprows=1, header=None, dtype=str)
    except EmptyDataError:
        print(f"⚠️ Empty file: {filename}")
        return
    except Exception as e:
        print(f"⚠️ Error reading {filename}: {e}")
        return

    # Get TXT headers
    with open(filename, 'r') as f:
        txt_header = f.readline().strip('\n').split('\t')

    # Get VCF headers safely
    vcf_file = filename.replace('.txt', '.vcf')
    vcf_header = []
    if os.path.exists(vcf_file):
        with open(vcf_file, 'r') as f:
            for line in f:
                if line.startswith('#CHROM'):
                    vcf_header = line.strip('\n').split('\t')
                    break
    else:
        print(f"⚠️ VCF file not found: {vcf_file}. Using default headers.")
        vcf_header = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE']

    # 1. FIX: Dynamically align columns to avoid "Length Mismatch" ValueError
    num_data_cols = df.shape[1]
    num_vcf_cols = len(vcf_header)
    
    new_columns = []
    # Fill from the left using txt_header, add generic padding if ANNOVAR inserted extra columns
    for i in range(num_data_cols - num_vcf_cols):
        if i < len(txt_header):
            new_columns.append(txt_header[i])
        else:
            new_columns.append(f"Padding_{i}")
            
    # Attach VCF headers exactly to the end
    new_columns.extend(vcf_header)
    df.columns = new_columns

    # 2. FIX: Safely find the end of the annotation slice
    if 'Otherinfo1' in df.columns:
        idx = list(df.columns).index('Otherinfo1')
        # Respect original logic: drop the column right before Otherinfo1
        end_of_annotation_idx = max(0, idx - 1) 
    else:
        end_of_annotation_idx = num_data_cols - num_vcf_cols

    # 3. FIX: Safely extract FORMAT keys without IndexErrors
    if 'FORMAT' not in df.columns:
        print(f"⚠️ FORMAT column not found in {filename}.")
        return
        
    format_col = df['FORMAT']
    sample_col = df.iloc[:, -1]  # The last column is always the sample data
    
    first_valid_fmt = format_col.dropna().iloc[0] if not format_col.dropna().empty else ""
    if not first_valid_fmt:
        print(f"⚠️ FORMAT column is entirely empty in {filename}.")
        return
        
    format_keys = str(first_valid_fmt).split(':')

    # Vectorized parsing
    sample_expanded = sample_col.str.split(':', expand=True)

    # 4. FIX: Safe extraction of DP and FREQ using column presence checks
    if 'DP' in format_keys:
        dp_idx = format_keys.index('DP')
        depth_col = sample_expanded[dp_idx].rename('Depth') if dp_idx in sample_expanded.columns else pd.Series(['.'] * len(df), name='Depth')
    else:
        depth_col = pd.Series(['.'] * len(df), name='Depth')

    if 'FREQ' in format_keys:
        freq_idx = format_keys.index('FREQ')
        vaf_col = sample_expanded[freq_idx].rename('VAF') if freq_idx in sample_expanded.columns else pd.Series(['.'] * len(df), name='VAF')
    else:
        vaf_col = pd.Series(['.'] * len(df), name='VAF')

    # Extract FILTER safely
    filter_col = df['FILTER'] if 'FILTER' in df.columns else pd.Series(['.'] * len(df), name='FILTER')

    # Build final dataframe
    final_df = pd.concat([
        df.iloc[:, :end_of_annotation_idx], 
        filter_col, 
        depth_col, 
        vaf_col
    ], axis=1)

    # Save
    final_df.to_csv(out_filename, sep='\t', index=False)
    print(f"✅ Processed {os.path.basename(filename)} → {out_filename}")

Run format function on all samples:

In [ ]:
### this is to run on multiple samples###
from tqdm import tqdm


# Load the report you generated earlier
report_path = os.path.join(output_5_vcfs_base_dir, "multianno_file_report.xlsx")
df = pd.read_excel(report_path)

# Filter only rows with valid file paths
df_valid = df[df["File_Path"].notna()]

print(f"🔎 Found {len(df_valid)} files to process.")

# Iterate over each file
for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Processing samples"):
    sample_name = row["Sample_Name"]
    file_path = row["File_Path"]
    try:
        # Call your function
        varscan_multianno_to_df_no_filter_fast(file_path)

        # Print success
        print(f"✅ Finished: {sample_name} → {file_path}")
    except Exception as e:
        # Print failure but keep looping
        print(f"❌ Error in {sample_name} ({file_path}): {e}")


## Merge all formatted files into a single one

In [ ]:
from tqdm import tqdm   

# Load the report you generated earlier
report_path = os.path.join(output_5_vcfs_base_dir, "multianno_file_report.xlsx")
df = pd.read_excel(report_path)

# Filter only rows with valid file paths
df_valid = df[df["File_Path"].notna()]

# reset index for easier iteration
df_valid.reset_index(drop=True, inplace=True)   

print(f"🔎 Found {len(df_valid)} files to process.")
    
# Iterate over each file, add sample_name as the 5th column and output to a single output file
for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Processing samples"):

    sample_name = row["Sample_Name"]
    file_path = row["File_Path"]
    try:
        # Read the formatted file into a DataFrame
        df_sample = pd.read_csv(file_path.replace('.txt', '.formatted.txt'), sep='\t', dtype=str)

        # Insert sample_name as the 6th column (index 5)
        df_sample.insert(5, 'Sample_Name', sample_name)

        df_sample = df_sample.rename(columns={'Chr': 'CHR',
        'Start': 'POS',
        'Ref': 'REF',
        'Alt': 'ALT'        
        })
        # Write to tab-delimnated file if idx == 0 and append for subsequent files
        if idx == 0:
            df_sample.to_csv(merged_varscan_file_path, sep='\t', index=False, header=True)
        else:
            df_sample.to_csv(merged_varscan_file_path, sep='\t', index=False, header=False, mode='a')
        
        # Print success
        print(f"✅ Finished: {idx}: {sample_name} → {file_path}")
    except Exception as e:
        # Print failure but keep looping
        print(f"❌ Error in {idx}: {sample_name} ({file_path}): {e}")


print(f"✅ Merged file saved to: {merged_varscan_file_path}")

## Apply Blood 2023 filtering on merged mutation table

In [ ]:
# This is the original R version - verified the python version is equivalent, no need to run the R one
# Call R mutation filtering and annotation script - RUN OUTSIDE under an R conda environment if it fails here
if False:
    import os   
    import subprocess

    # Define the path to the R script
    repo_root = get_git_root()
    os.chdir(Path(repo_root) / "Mutation_Curation" / "R")
    r_script =  "./FilterMutations.R" 

    # Define the command to run the R script
    command = f"conda run -n {conda_env} --no-capture-output {rscript_exe} {r_script} -input-file {merged_varscan_file_path} -output-dir {output_5_vcfs_base_dir} -run-name {seq_run}_arch -sample-column Sample_Name  -dups-input-file=False"
    print(command)
    # Run the command
    try:
        subprocess.run(command, check=True, shell=True)
        print("✅ R script executed successfully.")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error executing R script: {e}")


## Validate Python port of FilterMutations.R against the R output

In [ ]:
import sys
import numpy as np

repo_root = get_git_root()
sys.path.insert(0, str(Path(repo_root) / "utils"))
from filter_mutations import filter_mutations

py_result = filter_mutations(
    input_file=merged_varscan_file_path,
    output_dir=output_5_vcfs_base_dir,
    run_name=f"{seq_run}_arch_py",
    sample_column="Sample_Name",
    dups_input_file=False,
)

py_listed = pd.read_table(py_result["listed_fn"])
py_rest = pd.read_table(py_result["rest_fn"])


## Compare R vs python Blood 2023 filtering - done once, no need to rerun in each run

In [ ]:
if False:
    r_listed = pd.read_table(os.path.join(output_5_vcfs_base_dir, f"FilterMutations_{seq_run}_arch_listed.tsv"))
    r_rest = pd.read_table(os.path.join(output_5_vcfs_base_dir, f"FilterMutations_{seq_run}_arch_filtered_yet_not_listed.tsv"))

    key_cols = ["CHR", "POS", "REF", "ALT", "Sample_Name"]

    def compare(label, py_df, r_df):
        print(f"=== {label}: python {py_df.shape}, R {r_df.shape} ===")
        py_s = py_df.sort_values(key_cols).reset_index(drop=True)
        r_s = r_df.sort_values(key_cols).reset_index(drop=True)
        if len(py_s) != len(r_s):
            print("  ROW COUNT MISMATCH")
            return
        shared_cols = [c for c in py_s.columns if c in r_s.columns]
        n_diff_cols = 0
        for c in shared_cols:
            a, b = py_s[c], r_s[c]
            if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
                same = np.isclose(a.astype(float), b.astype(float), equal_nan=True, atol=1e-6)
            else:
                same = (a.astype(str) == b.astype(str)) | (a.isna() & b.isna())
            if not same.all():
                n_diff_cols += 1
                print(f"  column '{c}' differs in {(~same).sum()} row(s)")
        print(f"  {'MATCH' if n_diff_cols == 0 else 'MISMATCH'}: {n_diff_cols} differing column(s) out of {len(shared_cols)} shared columns\n")

    compare("listed", py_listed, r_listed)
    compare("rest", py_rest, r_rest)


# Some plots

In [ ]:
valid_samples = pd.read_csv(os.path.join(output_5_vcfs_base_dir, samples_for_processing_ifn))

df = pd.read_table(py_result["listed_fn"], sep='\t')
rest_df = pd.read_table(py_result["rest_fn"], sep='\t')
print(df.shape)
samples_with_mutations = len(df['Sample_Name'].unique())
print(f"🔎 Found {samples_with_mutations}/{valid_samples.shape[0]} samples with mutations")

sample_mut_counts = df['Sample_Name'].value_counts().sort_index().reset_index()
sample_mut_counts.columns = ['Sample_Name', 'Count']

plt.figure(figsize=(10, 6))
sns.histplot(data=sample_mut_counts, x='Count', color='steelblue', bins=sample_mut_counts['Count'].max())
plt.xlabel('Number of Mutations')
plt.ylabel('Number of Samples')
plt.title('Number of Mutations per Sample')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Group df by 'CHR', 'POS', and 'Gene.refGene'
df_grouped = (
    df.groupby(['CHR', 'POS', 'Gene.refGene', 'Func.refGene', 'ExonicFunc.refGene'])
      .agg(
          number_of_samples=('Sample_Name', 'nunique'),
          avg_vaf=('avg_VAF', 'mean'),
          max_vaf=('avg_VAF', 'max')
      )
      .reset_index()
)


plt.figure(figsize=(12, 7))
ax = sns.scatterplot(
    data=df_grouped,
    x='number_of_samples',
    y='avg_vaf',
    s=80,
    color='skyblue',
    edgecolor='gray'
)

# Add text labels for each point
df_grouped_to_show = df_grouped[(df_grouped['number_of_samples'] >= 3) | (df_grouped['avg_vaf'] > 0.5)]
for i, row in df_grouped_to_show.iterrows():
    ax.text(
        row['number_of_samples'],
        row['avg_vaf'],
        str(row['Gene.refGene']),
        fontsize=10,
        ha='center',
        va='top'
    )

plt.xlabel('Number of Samples')
plt.ylabel('Average VAF')
plt.title('Number of Samples vs. Average VAF (labeled by gene)')
plt.tight_layout()
plt.show()

In [ ]:
df_grouped_to_show = df_grouped_to_show.sort_values('number_of_samples', ascending=False)
display(df_grouped_to_show)
s = rest_df[(rest_df['Gene.refGene'] == 'MPL') & (rest_df['avg_VAF'] >= 0.05) & (rest_df['mean_depth'] >= 20)]
# s = df[df['Gene.refGene'] == 'MPL']
display(s)
for v in s.index:
     print(v)

In [ ]:
df_u = df.drop_duplicates(subset=['Sample_Name', 'Gene.refGene'])
gene_counts = df_u['Gene.refGene'].value_counts().sort_index().reset_index()

gene_counts.columns = ['Gene', 'Count']
gene_counts = gene_counts.sort_values('Count', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=gene_counts, x='Gene', y='Count', color='darkred')
plt.xlabel('Gene')
plt.ylabel('Number of samples')
plt.title('Number of samples with mutations per Gene')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Remove recurrent mutations without cosmic or clinvar support

In [ ]:
import re

if filter_recurrent_without_support:
    # Filter out recurrent mutations (>=2 samples) that lack haematopoietic COSMIC or ClinVar support
    variant_cols = ['CHR', 'POS', 'REF', 'ALT', 'Gene.refGene', 'Selected.AA.change']

    force_include_variants = []
    # Specific mutations to always keep regardless of recurrence criteria
    # Add dicts with any subset of variant_cols keys, e.g. {'Gene.refGene': 'ASXL1', 'Selected.AA.change': 'G642fs'}
    # force_include_variants = [
    #     {'Gene.refGene': 'ASXL1', 'Selected.AA.change': 'G642fs'},
    # ]

    def extract_total_cosmic_count(cosmic_str):
        if pd.isna(cosmic_str) or cosmic_str == '.':
            return 0
        occ_match = re.search(r'OCCURENCE=([^;]+)', cosmic_str)
        if not occ_match:
            return 0
        counts = re.findall(r'(\d+)\(', occ_match.group(1))
        return sum(int(c) for c in counts)

    # Count how many samples each variant appears in
    df = pd.read_csv(py_result["listed_fn"], sep='\t')
    variant_sample_counts = df.groupby(variant_cols)['Sample_Name'].nunique().reset_index(name='Sample_Count')
    df_with_counts = pd.merge(df, variant_sample_counts, on=variant_cols, how='left')

    # Separate single-sample and multi-sample variants
    non_recurrent = df_with_counts[df_with_counts['Sample_Count'] == 1].copy()
    recurrent = df_with_counts[df_with_counts['Sample_Count'] >= 2].copy()

    # Classify recurrent variants by COSMIC haematopoietic and ClinVar support
    recurrent_info = recurrent[variant_cols + ['cosmic103', 'CLNDN']].drop_duplicates(subset=variant_cols)
    recurrent_info['COSMIC_Total_Count'] = recurrent_info['cosmic103'].apply(extract_total_cosmic_count)
    recurrent_info['COSMIC_in_haematopoetic'] = recurrent_info['cosmic103'].str.contains(
        'haematopoietic_and_lymphoid_tissue', na=False)
    recurrent_info['in_clinvar'] = ~recurrent_info['CLNDN'].isin(['.', 'not_provided', 'not_specified'])
    recurrent_info['Keep'] = (
        (recurrent_info['COSMIC_Total_Count'] > 0) & recurrent_info['COSMIC_in_haematopoetic']
    ) | recurrent_info['in_clinvar']
    print(recurrent_info['Keep'].value_counts())

    # Force-include specific mutations
    for spec in force_include_variants:
        mask = pd.Series([True] * len(recurrent_info), index=recurrent_info.index)
        for col, val in spec.items():
            if col in recurrent_info.columns:
                mask &= (recurrent_info[col] == val)
        recurrent_info.loc[mask, 'Keep'] = True

    # Keep only recurrent variants with haematopoietic COSMIC or ClinVar support
    keep_variants = recurrent_info[recurrent_info['Keep']][variant_cols]
    filtered_recurrent = pd.merge(recurrent, keep_variants, on=variant_cols, how='inner')


    # Combine non-recurrent with supported recurrent variants and save
    final_df = pd.concat([non_recurrent, filtered_recurrent], ignore_index=True)
    final_df = final_df.drop(columns=['Sample_Count'])

    final_df.to_csv(no_recurrent_output_path, index=False, sep='\t')

    n_removed = len(df) - len(final_df)
    print(f'Removed {n_removed} recurrent variants without haematopoietic COSMIC or ClinVar support')
    print(f'Remaining: {len(final_df)} mutations across {final_df["Sample_Name"].nunique()} samples')
else:
    # Step skipped (filter_recurrent_without_support = False in the parameters cell):
    # carry the listed mutations through untouched.
    final_df = pd.read_csv(py_result["listed_fn"], sep='\t')
    print('Recurrence filter skipped (filter_recurrent_without_support=False)')
    print(f'Carrying through all {len(final_df)} mutations across {final_df["Sample_Name"].nunique()} samples')


# VAF filter and remove excluded mutations

In [ ]:

# Apply final VAF threshold (per-gene override via gene_min_vaf_overrides, else min_final_vaf) and drop specific known artefacts/germline calls
effective_min_vaf = final_df['Gene.refGene'].map(gene_min_vaf_overrides).fillna(min_final_vaf)
vaf_filtered = final_df[final_df['VAF'] > effective_min_vaf].copy()
print(f'After VAF filter (min_final_vaf={min_final_vaf}, overrides={gene_min_vaf_overrides}): {len(vaf_filtered)}')

# to_remove = pd.DataFrame({
#     'Gene.refGene': ['CEBPA', 'NF1', 'CEBPA'],
#     'Selected.AA.change': ['E334X', 'M2010fs', 'N321fs']
# })
# exclusion_keys = ['Gene.refGene', 'Selected.AA.change']
to_remove = pd.DataFrame({
    'Gene.refGene': ['STAG2',   'STAG2',  'PHF6',    'PHF6',    'BCORL1',  'BCORL1',  'BCORL1',   'BCORL1',   'WT1',    'WT1',    'WT1'],
    'POS':          [124066174, 124050219, 134378011, 134378017, 130015631, 130015637,  130015640, 130015640, 32434878, 32434880, 32434890],
    'REF' :         ['G',       '-',       '-',       '-',       '-',       'C',        'C',       '-',       '-',      'A',      'G'],
    'ALT' :         ['C',       'A',       'G',       'C',       'G',       '-',        '-',       'A',       'T',      '-',      '-'],
})
exclusion_keys = ['Gene.refGene', 'POS', 'REF', 'ALT']

mask = ~vaf_filtered.set_index(exclusion_keys).index.isin(to_remove.set_index(exclusion_keys).index)
vaf_filtered = vaf_filtered[mask]
print(f'After removing excluded mutations: {len(vaf_filtered)}')

vaf_filtered.to_csv(master_output_path, index=False, sep='\t')

print(f'Removed {len(final_df) - len(vaf_filtered)} rows via VAF filter and manual exclusions')
print(f'Remaining: {len(vaf_filtered)} mutations across {vaf_filtered["Sample_Name"].nunique()} samples')
print(f'Saved to: {master_output_path}')


# Detect CALR Type-1 (del52) from FASTQ — bsub mode

In [ ]:
import sys
import subprocess

repo_root = get_git_root()

calr_type1_script = os.path.join(repo_root, "utils", "CALR_type1_from_fastq.py")

cmd = [
    sys.executable,
    calr_type1_script,
    "--submit",
    "--fastq_dir", fastq_dir,
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=output_5_vcfs_base_dir)

print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)

# Detect CALR Type-2 (ins5) from BAM — bsub mode

In [ ]:
calr_type2_script = os.path.join(repo_root, "utils", "CALR_type2_from_bam.py")

cmd = [
    sys.executable,
    calr_type2_script,
    "--bam_dir", mapped_dir,
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=output_5_vcfs_base_dir)

print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)

# Collect CALR results

In [ ]:


dfs1 = []
dfs2 = []
valid_samples = pd.read_csv(os.path.join(output_5_vcfs_base_dir, samples_for_processing_ifn))
for sample_name in valid_samples['Sample']:
    type1_file = os.path.join(output_5_vcfs_base_dir, "calr_results", f"{sample_name}_CALR.tsv")
    type2_file = os.path.join(output_5_vcfs_base_dir, "calr_results", f"{sample_name}_CALR_type2.tsv")

    if os.path.exists(type1_file):
        df1 = pd.read_csv(type1_file, sep='\t')
        df1['Sample_Name'] = sample_name
        dfs1.append(df1)
    else:
        print(f'Alert: {type1_file} does not exist for sample {sample_name}.')
    
    if os.path.exists(type2_file):
        df2 = pd.read_csv(type2_file, sep='\t')
        df2['Sample_Name'] = sample_name
        dfs2.append(df2)
    else:
        print(f'Alert: {type2_file} does not exist for sample {sample_name}.')

calr_type1_df = pd.concat(dfs1, ignore_index=True)
calr_type1_df.to_csv(calr_type1_results_file, sep='\t', index=False)
calr_type2_df = pd.concat(dfs2, ignore_index=True)
calr_type2_df.to_csv(calr_type2_results_file, sep='\t', index=False)


# Filter by Denver normals

Allow mutations that either have not appeared in Denver or appeared and are in the list of valid Denver mutations (seperate list for SNVs and indels).

In [ ]:
# Read filtered df (allowing re-run of this cell without re-running the filtering steps above
vaf_filtered = pd.read_csv(master_output_path, sep='\t')

denver_key_cols = ['CHR', 'POS', 'REF', 'ALT']

# Step 1: check if mutations were observed in Denver
query_exe = os.path.join(denver_base_dir, "variant_index", "query.py")
is_in_denver_path = master_output_path.replace('.tsv', '_is_in_denver.tsv')

cmd = [
    sys.executable,
    query_exe,
    "--tsv", master_output_path,
    "-o", is_in_denver_path,
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
denver_df = pd.read_csv(is_in_denver_path, sep='\t')
denver_df = denver_df.rename(columns={'CHROM': 'CHR'})

# Step 2: check if mutations are in Denver's filtered passed SNVs
repo_root = get_git_root()

denver_filtered_SNVs_path = os.path.join(Path(repo_root) / 'notebooks' / 'denver' / 'out' / 'Poisson_annotated_merged_POOLED_RESTART_180826_filtered_SNVs.tsv')
denver_filtered_SNVs = pd.read_csv(denver_filtered_SNVs_path, sep='\t')
denver_filtered_SNVs = denver_filtered_SNVs[denver_key_cols]
denver_filtered_SNVs['Valid SNV'] = True
denver_filtered_SNVs = denver_filtered_SNVs.drop_duplicates()

denver_df = denver_df.merge(
    denver_filtered_SNVs[denver_key_cols + ['Valid SNV']],
    how='left',
    on=denver_key_cols)
denver_df['Valid SNV'] = denver_df['Valid SNV'].fillna(False)

# Step 2b: Same for indels
denver_filtered_Indels_path = os.path.join(Path(repo_root) / 'notebooks' / 'denver' / 'out' / 'filtering_Indels_annotated_varscan_filtered.tsv')
denver_filtered_Indels = pd.read_csv(denver_filtered_Indels_path, sep='\t')
denver_filtered_Indels = denver_filtered_Indels[denver_key_cols]
denver_filtered_Indels['Valid Indel'] = True
denver_filtered_Indels = denver_filtered_Indels.drop_duplicates()

denver_df = denver_df.merge(
    denver_filtered_Indels[denver_key_cols + ['Valid Indel']],
    how='left',
    on=denver_key_cols)
denver_df['Valid Indel'] = denver_df['Valid Indel'].fillna(False)

valid_by_denver = ((denver_df['match'] == 'none') 
| (denver_df['Valid SNV']) 
| (denver_df['Valid Indel']))
denver_df['Valid by Denver'] = valid_by_denver
valid_by_denver_count = valid_by_denver.sum()
print(f"✅ {valid_by_denver_count}/{len(denver_df)} mutations are either not in Denver or are valid SNVs/Indels in Denver")

# Add the Denver validation results to the filtered VAF DataFrame
vaf_filtered_with_denver = vaf_filtered.merge(
    denver_df,
    how='left',
    on=denver_key_cols,
    suffixes=('', '_denver')
)
vaf_filtered_with_denver['is_valid'] = vaf_filtered_with_denver['Valid by Denver'] | (vaf_filtered_with_denver['Has.listed.match_type'] == 'narrow')
vaf_filtered_with_denver.to_csv(master_output_with_denver_path, index=False, sep='\t')

denver_filtered = vaf_filtered_with_denver[vaf_filtered_with_denver['is_valid']]
denver_filtered.to_csv(denver_filtered_output_path, index=False, sep='\t')


# Alert on target mutations

In [ ]:
import sys
import subprocess

mutation_alert_script = os.path.abspath(os.path.join('..', 'utils', 'mutation_alert.py'))
mutation_alert_config = os.path.abspath(os.path.join('..', 'utils', 'mutation_alert_config.yaml'))


cmd = [
    sys.executable,
    mutation_alert_script,
    "--mutations-file", denver_filtered_output_path,
    "--config", mutation_alert_config,
    "--run-name", seq_run,
    "--calr-type1-file", calr_type1_results_file,
    "--calr-type2-file", calr_type2_results_file,
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)